# Multi-Game Player and Lineup Performance Metrics

## Objective

This notebook computes performance metrics using the validated multi-game stint dataset.

The goal is to evaluate:

- Player impact
- Lineup performance
- Player pair synergy

These metrics provide the foundation for downstream modeling, optimization, and dashboard development.

## Approach

Metrics are derived from stint-level scoring data and normalized by playing time or stint frequency to allow fair comparison across players and lineups.

In [1]:
import pandas as pd
import numpy as np

## Load validated multi-game stint dataset

In [2]:
stints_df = pd.read_pickle("../data/interim/stints_multigame_validated.pkl")

stints_df.shape

(6293, 22)

## Create player-level records from stints

Each stint contributes to the performance of all players on the court.

In [3]:
player_records = []

for _, row in stints_df.iterrows():
    duration = row["end_eventnum"] - row["start_eventnum"]

    # Home players
    for player in row["home_lineup"]:
        player_records.append({
            "player_id": str(player),
            "game_id": row["game_id"],
            "duration": duration,
            "net_points": row["home_points"] - row["away_points"]
        })

    # Away players
    for player in row["away_lineup"]:
        player_records.append({
            "player_id": str(player),
            "game_id": row["game_id"],
            "duration": duration,
            "net_points": row["away_points"] - row["home_points"]
        })

player_df = pd.DataFrame(player_records)

player_df.shape

(62930, 4)

## Compute player impact metrics

In [4]:
player_impact = (
    player_df
    .groupby("player_id")
    .agg(
        total_net_points=("net_points", "sum"),
        total_duration=("duration", "sum"),
        total_stints=("duration", "count")
    )
    .reset_index()
)

player_impact["impact_per_60_sec"] = (
    player_impact["total_net_points"] / player_impact["total_duration"]
)

player_impact.sort_values("impact_per_60_sec", ascending=False).head()

,player_id,total_net_points,total_duration,total_stints,impact_per_60_sec
178,2061,11,137,5,0.080292
297,693,34,434,36,0.078341
200,2143,8,109,8,0.073394
280,56,101,1589,102,0.063562
167,2045,50,849,70,0.058893


In [5]:
player_impact["stints_per_game"] = (
    player_df.groupby("player_id")["game_id"].nunique()
)

reliable_player_impact = player_impact[
    (player_impact["total_duration"] >= 200) &
    (player_impact["stints_per_game"] >= 3)
].copy()

In [6]:
player_impact["impact_per_60_sec"].describe()

count    383.000000
mean      -0.003823
std        0.030064
min       -0.166667
25%       -0.017734
50%       -0.001280
75%        0.011964
max        0.080292
Name: impact_per_60_sec, dtype: float64

## Filter reliable player sample


In [7]:
reliable_player_impact = player_impact[
    player_impact["total_duration"] >= 200
].copy()

reliable_player_impact.shape

(363, 6)

## Compute lineup performance

In [8]:
# Convert lineup lists to tuples so they can be grouped
stints_df["home_lineup_tuple"] = stints_df["home_lineup"].apply(tuple)
stints_df["away_lineup_tuple"] = stints_df["away_lineup"].apply(tuple)

lineup_perf = (
    stints_df
    .groupby(["home_lineup_tuple", "away_lineup_tuple"])
    .agg(
        total_home_points=("home_points", "sum"),
        total_away_points=("away_points", "sum"),
        total_duration=("duration_events", "sum"),
        total_stints=("duration_events", "count")
    )
    .reset_index()
)

lineup_perf["net_points"] = (
    lineup_perf["total_home_points"] - lineup_perf["total_away_points"]
)

lineup_perf["net_per_60"] = (
    lineup_perf["net_points"] / lineup_perf["total_duration"]
) * 60

lineup_perf.head()

,home_lineup_tuple,away_lineup_tuple,total_home_points,total_away_points,total_duration,total_stints,net_points,net_per_60
0,"(1000, 1005, 1508, 1509, 165)","(1497, 1725, 1887, 210, 324)",-86,-74,55,1,-12,-13.090909
1,"(1000, 1005, 1508, 1509, 165)","(1497, 1725, 210, 324, 417)",6,7,27,3,-1,-2.222222
2,"(1000, 1005, 1508, 165, 1749)","(1032, 1501, 208, 281, 283)",-66,-73,70,3,7,6.000000
3,"(1000, 1005, 1508, 165, 1749)","(124, 1444, 1513, 1715, 468)",2,8,23,3,-6,-15.652174
4,"(1000, 1005, 1508, 165, 1749)","(124, 1513, 1715, 185, 57)",-101,-68,63,2,-33,-31.428571


In [9]:
reliable_lineups = lineup_perf[
    lineup_perf["total_duration"] >= 5
].copy()

## Compute player pair synergy

In [10]:
from itertools import combinations

pair_records = []

for _, row in stints_df.iterrows():
    duration = row["end_eventnum"] - row["start_eventnum"]

    # Home pairs
    for p1, p2 in combinations(row["home_lineup"], 2):
        pair_records.append({
            "player_1": str(p1),
            "player_2": str(p2),
            "net_points": row["home_points"] - row["away_points"],
            "duration": duration
        })

    # Away pairs
    for p1, p2 in combinations(row["away_lineup"], 2):
        pair_records.append({
            "player_1": str(p1),
            "player_2": str(p2),
            "net_points": row["away_points"] - row["home_points"],
            "duration": duration
        })

pair_df = pd.DataFrame(pair_records)

In [11]:
pair_perf = (
    pair_df
    .groupby(["player_1", "player_2"])
    .agg(
        total_net_points=("net_points", "sum"),
        total_duration=("duration", "sum"),
        total_stints=("duration", "count")
    )
    .reset_index()
)

pair_perf["impact_per_60"] = (
    pair_perf["total_net_points"] / pair_perf["total_duration"]
)

In [12]:
reliable_pairs = pair_perf[
    pair_perf["total_duration"] >= 200
].copy()

## Save outputs

In [13]:
player_impact.to_csv("../data/processed/player_impact.csv", index=False)
reliable_player_impact.to_csv("../data/processed/reliable_player_impact.csv", index=False)

lineup_perf.to_csv("../data/processed/lineup_performance.csv", index=False)
reliable_lineups.to_csv("../data/processed/reliable_lineups.csv", index=False)

pair_perf.to_csv("../data/processed/pair_performance.csv", index=False)
reliable_pairs.to_csv("../data/processed/reliable_pair_performance.csv", index=False)

## Key findings

The multi-game dataset enables more stable and meaningful performance metrics than the original single-game prototype.

- Player impact reflects consistent on-court contributions across games
- Lineup performance captures interactions between players
- Pair synergy highlights effective player combinations

These metrics provide the foundation for predictive modeling and optimization.

## Next step

The next notebook will use these metrics to build predictive models and evaluate lineup performance.